# 04 — Chaves, cardinalidade, filiação no tempo

Requer `03` e seu manifesto congelado. Testes sobre amostras não provam unicidade global. Nenhuma junção entre Casas é feita por número, nome ou ementa.

In [1]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'validation_utils.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from validation_utils import (Probe, OUT, CHECKS, MANIFEST, MANIFEST_FIELDS, api_camara, api_senado,
                              record, manifest_rows, save_manifest, rows_camara, count_values, csv_head)
pd.set_option('display.max_colwidth', 90)

from collections import Counter
probe=Probe('04',max_calls=100)
manifest=manifest_rows()
assert manifest, 'Execute 03 para congelar o manifesto.'
projects=[r for r in manifest if r['entidade']=='projeto']
print('Projetos no manifesto:',len(projects),'| limite de detalhe:',min(12,len(projects)))

Projetos no manifesto: 30 | limite de detalhe: 12


## IDs de projeto, campos nulos e detalhes

In [2]:
sample=projects[:12]
details=[]; failures=[]
for row in sample:
    r=probe.json(row['url'])
    if not r.ok: failures.append((row['fonte'],row['id'],r.error)); continue
    d=r.data.get('dados',{}) if row['fonte']=='camara' else r.data
    details.append({'fonte':row['fonte'],'id_manifesto':row['id'],'id_resposta':str(d.get('id','')),
                    'tipo':d.get('siglaTipo') or d.get('sigla'),'ano_manifesto':row['ano'],
                    'data':d.get('dataApresentacao') or (d.get('documento') or {}).get('dataApresentacao')})
frame=pd.DataFrame(details)
display(frame)
counts={'consultados':len(sample),'detalhes':len(frame),'IDs_distintos':len(set((x['fonte'],x['id_resposta']) for x in details)),
        'id_divergente':sum(x['id_manifesto']!=x['id_resposta'] for x in details),'data_nula':sum(not x['data'] for x in details)}
record('04','ids_projeto','ambas','IDs do manifesto → endpoints de detalhe',len(frame),
       'ID nativo do detalhe coincide com manifesto e não repete na amostra',counts,
       'PASSOU' if not failures and counts['id_divergente']==0 and counts['IDs_distintos']==len(frame) else 'INCONCLUSIVO',str(failures[:3]))

,fonte,id_manifesto,id_resposta,tipo,ano_manifesto,data
0,camara,2647384,2647384,PL,2026,2026-10-08T17:21
1,camara,2647345,2647345,PL,2026,2026-10-07T16:07
2,camara,2599994,2599994,PLP,2026,2026-02-02T09:22
3,camara,2600175,2600175,PLP,2026,2026-02-02T15:05
4,camara,2604173,2604173,PEC,2026,2026-02-24T15:48
5,camara,2623400,2623400,PEC,2026,2026-05-11T14:49
6,senado,9101538,9101538,PL,2026,2026-09-01
7,senado,9101917,9101917,PL,2026,2026-09-01
8,senado,9104876,9104876,PLP,2026,2026-09-02
9,senado,9105948,9105948,PLP,2026,2026-09-03


{'notebook': '04',
 'check': 'ids_projeto',
 'fonte': 'ambas',
 'url_parametros': 'IDs do manifesto → endpoints de detalhe',
 'coletado_em_utc': '2026-10-09T14:22:09+00:00',
 'amostra_n': 12,
 'regra': 'ID nativo do detalhe coincide com manifesto e não repete na amostra',
 'contagens': '{"IDs_distintos": 12, "consultados": 12, "data_nula": 0, "detalhes": 12, "id_divergente": 0}',
 'estado': 'PASSOU',
 'motivo': '[]'}

## Autoria e tramitação da Câmara: chaves compostas, órfãos e multiplicação

In [3]:
cam=[x for x in projects if x['fonte']=='camara'][:4]
relation_rows=[]; errors=[]
for p in cam:
    for relation in ('autores','tramitacoes'):
        r=probe.json(api_camara(f"proposicoes/{p['id']}/{relation}"),limit=2_000_000)
        if not r.ok: errors.append(f"{p['id']}/{relation}: {r.error}"); continue
        for item in rows_camara(r.data):
            key=(item.get('uri'),item.get('ordemAssinatura')) if relation=='autores' else (item.get('sequencia'),)
            relation_rows.append({'projeto':p['id'],'relacao':relation,'chave_local':str(key),'sem_chave':all(v is None for v in key)})
rel=pd.DataFrame(relation_rows)
display(rel.groupby(['projeto','relacao']).size().rename('linhas').reset_index() if not rel.empty else rel)
dups=int(rel.duplicated(['projeto','relacao','chave_local']).sum()) if not rel.empty else 0
record('04','relacoes_camara','camara','/proposicoes/{id}/autores e /tramitacoes',len(rel),
       'Unicidade das chaves locais e filhos com pai conhecido na amostra',
       {'projetos':len(cam),'filhos':len(rel),'duplicatas_chave_candidata':dups,'sem_chave':int(rel['sem_chave'].sum()) if not rel.empty else 0},
       'INCONCLUSIVO' if errors or dups else 'PASSOU',str(errors[:3])+'; unicidade global não demonstrada')

,projeto,relacao,linhas
0,2599994,autores,1
1,2599994,tramitacoes,1
2,2600175,autores,1
3,2600175,tramitacoes,1
4,2647345,autores,1
5,2647345,tramitacoes,1
6,2647384,autores,1
7,2647384,tramitacoes,1


{'notebook': '04',
 'check': 'relacoes_camara',
 'fonte': 'camara',
 'url_parametros': '/proposicoes/{id}/autores e /tramitacoes',
 'coletado_em_utc': '2026-10-09T14:22:17+00:00',
 'amostra_n': 8,
 'regra': 'Unicidade das chaves locais e filhos com pai conhecido na amostra',
 'contagens': '{"duplicatas_chave_candidata": 0, "filhos": 8, "projetos": 4, "sem_chave": 0}',
 'estado': 'PASSOU',
 'motivo': '[]; unicidade global não demonstrada'}

## Retrato atual: composição por partido e UF nas duas Casas

In [4]:
deputies=[]; page_errors=[]
for page in range(1,8):
    r=probe.json(api_camara('deputados'),{'itens':100,'pagina':page,'ordem':'ASC','ordenarPor':'id'})
    if not r.ok: page_errors.append(r.error); break
    items=rows_camara(r.data)
    deputies.extend(items)
    if len(items)<100: break
sen_current=probe.json(api_senado('senador/lista/atual.json'))
senators=(sen_current.data or {}).get('ListaParlamentarEmExercicio',{}).get('Parlamentares',{}).get('Parlamentar',[]) if sen_current.ok else []
dc=Counter(str(x.get('id')) for x in deputies)
sc=Counter(str(x.get('IdentificacaoParlamentar',{}).get('CodigoParlamentar')) for x in senators)
display(pd.DataFrame([{'casa':'camara','linhas':len(deputies),'IDs_distintos':len(dc),
                       'partido_nulo':sum(not x.get('siglaPartido') for x in deputies),'UF_nula':sum(not x.get('siglaUf') for x in deputies)},
                      {'casa':'senado','linhas':len(senators),'IDs_distintos':len(sc),
                       'partido_nulo':sum(not x.get('IdentificacaoParlamentar',{}).get('SiglaPartidoParlamentar') for x in senators),
                       'UF_nula':sum(not x.get('IdentificacaoParlamentar',{}).get('UfParlamentar') for x in senators)}]))
good=not page_errors and sen_current.ok and len(deputies)>0 and len(senators)>0 and len(dc)==len(deputies) and len(sc)==len(senators)
record('04','composicao_atual','ambas','/deputados e /senador/lista/atual.json',len(deputies)+len(senators),
       'Listagens atuais paginadas sem IDs duplicados e com partido/UF observados',
       {'deputados':len(deputies),'senadores':len(senators),'duplicatas_camara':len(deputies)-len(dc),
        'duplicatas_senado':len(senators)-len(sc)},'PASSOU' if good else 'INCONCLUSIVO',
       'Apenas retrato atual; histórico exige vigência. '+str(page_errors[:2])+' '+sen_current.error)

,casa,linhas,IDs_distintos,partido_nulo,UF_nula
0,camara,513,513,0,0
1,senado,81,81,0,0


{'notebook': '04',
 'check': 'composicao_atual',
 'fonte': 'ambas',
 'url_parametros': '/deputados e /senador/lista/atual.json',
 'coletado_em_utc': '2026-10-09T14:22:24+00:00',
 'amostra_n': 594,
 'regra': 'Listagens atuais paginadas sem IDs duplicados e com partido/UF observados',
 'contagens': '{"deputados": 513, "duplicatas_camara": 0, "duplicatas_senado": 0, "senadores": 81}',
 'estado': 'PASSOU',
 'motivo': 'Apenas retrato atual; histórico exige vigência. [] '}

## Senado: mandato e filiações têm vigência; partido atual não serve para voto antigo

In [5]:
senator_id='5672'
responses={kind:probe.json(api_senado(f'senador/{senator_id}/{kind}.json')) for kind in ('mandatos','filiacoes')}
for kind,r in responses.items():
    if r.ok:
        print(kind,'envelope:',list(r.data),'parlamentar:',list(next(iter(r.data.values())).get('Parlamentar',{})))
    else: print(kind,r.error)
aff=responses['filiacoes']
filiacoes=[]
if aff.ok:
    parlamentar=next(iter(aff.data.values())).get('Parlamentar',{})
    raw=parlamentar.get('Filiacoes',{}).get('Filiacao',[]) if isinstance(parlamentar.get('Filiacoes'),dict) else []
    filiacoes=raw if isinstance(raw,list) else [raw]
display(pd.json_normalize(filiacoes).head(8) if filiacoes else pd.DataFrame())
party_at={}
for date in ('2023-10-21','2026-09-01'):
    valid=[x for x in filiacoes if (x.get('DataFiliacao') or '9999')<=date<= (x.get('DataDesfiliacao') or '9999')]
    party_at[date]=[x.get('Partido',{}).get('SiglaPartido') for x in valid]
print('Partido nas datas amostradas:',party_at)
record('04','filiacao_temporal','senado',api_senado(f'senador/{senator_id}/filiacoes.json'),len(filiacoes),
       'Identificar intervalos de filiação e partido vigente em duas datas',
       {'filiacoes':len(filiacoes),'partido_por_data':party_at},
       'PASSOU' if all(len(v)==1 for v in party_at.values()) else 'INCONCLUSIVO',
       'Um senador não valida todos os mandatos/licenças nem todas as mudanças históricas')

mandatos envelope: ['MandatoParlamentar'] parlamentar: ['Codigo', 'Nome', 'Mandatos']
filiacoes envelope: ['FiliacaoParlamentar'] parlamentar: ['Codigo', 'Nome', 'Filiacoes']


,DataFiliacao,Partido.CodigoPartido,Partido.SiglaPartido,Partido.NomePartido,DataDesfiliacao
0,2025-11-12,521,REPUBLICANOS,Republicanos,NaN
1,2022-02-24,582,UNIÃO,União Brasil,2025-11-10
2,2017-08-01,554,DEM,Democratas,2022-02-23
3,2015-02-01,521,REPUBLICANOS,Republicanos,2017-07-31


Partido nas datas amostradas: {'2023-10-21': ['UNIÃO'], '2026-09-01': ['REPUBLICANOS']}


{'notebook': '04',
 'check': 'filiacao_temporal',
 'fonte': 'senado',
 'url_parametros': 'https://legis.senado.leg.br/dadosabertos/senador/5672/filiacoes.json',
 'coletado_em_utc': '2026-10-09T14:22:26+00:00',
 'amostra_n': 4,
 'regra': 'Identificar intervalos de filiação e partido vigente em duas datas',
 'contagens': '{"filiacoes": 4, "partido_por_data": {"2023-10-21": ["UNIÃO"], "2026-09-01": ["REPUBLICANOS"]}}',
 'estado': 'PASSOU',
 'motivo': 'Um senador não valida todos os mandatos/licenças nem todas as mudanças históricas'}

## Cruzamento entre Casas: somente identificador oficial resolvido

In [6]:
sen=[x for x in projects if x['fonte']=='senado'][:6]
candidates=[]
for p in sen:
    r=probe.json(p['url'])
    if not r.ok: continue
    d=r.data
    candidates.append({'idProcesso':d.get('id'),'identificacao':d.get('identificacao'),
                       'identificacaoExterna':d.get('identificacaoExterna'),
                       'idProcessoCasaInicial':d.get('idProcessoCasaInicial'),
                       'siglaCasaIniciadora':d.get('siglaCasaIniciadora')})
display(pd.DataFrame(candidates))
explicit=sum(bool(x['identificacaoExterna'] or x['idProcessoCasaInicial']) for x in candidates)
record('04','vinculo_entre_casas','ambas','campos de origem nos detalhes do Senado',len(candidates),
       'Usar somente identificador cruzado documentado e resolvido na Câmara',
       {'candidatos_com_campo':explicit,'vinculos_resolvidos':0},'INCONCLUSIVO',
       'Campos observados não bastam sem confirmar identificador nativo da Câmara; nenhum join automático')

,idProcesso,identificacao,identificacaoExterna,idProcessoCasaInicial,siglaCasaIniciadora
0,9101538,PL 5241/2026,{},9101538,SF
1,9101917,PL 5245/2026,{},9101917,SF
2,9104876,PLP 247/2026,{},9104876,SF
3,9105948,PLP 74/2026,{},9105947,CD
4,8986903,PEC 1/2026,{},8986903,SF
5,9104654,PEC 17/2026,{},9104654,SF


{'notebook': '04',
 'check': 'vinculo_entre_casas',
 'fonte': 'ambas',
 'url_parametros': 'campos de origem nos detalhes do Senado',
 'coletado_em_utc': '2026-10-09T14:22:32+00:00',
 'amostra_n': 6,
 'regra': 'Usar somente identificador cruzado documentado e resolvido na Câmara',
 'contagens': '{"candidatos_com_campo": 6, "vinculos_resolvidos": 0}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Campos observados não bastam sem confirmar identificador nativo da Câmara; nenhum join automático'}